# 02 Embedding-space geometry and hubness

This notebook is self-contained. It downloads NFCorpus from the Hugging Face Hub, encodes the corpus, and calculates all numbers and figures. It does not read `results/`, `cache/`, or a `silent_rag` module. You need only the pip packages in the next cell. On a laptop CPU, the run time is approximately 10 to 20 minutes. Most of this time is for the encoder (approximately 3,600 documents) and the t-SNE runs.

**Questions**
1. Where are the queries and the documents in the embedding space?
2. Can we trust the t-SNE pictures?
3. Is the hubness larger than the hubness that high-dimensional geometry alone causes?

**Method**
This notebook calculates PCA variance, query-document cosines, t-SNE trustworthiness for different seeds and perplexities, and hubness. For hubness, it uses subsamples of equal size and compares them with isotropic and covariance-matched null vectors.

**Scope.** This notebook uses only NFCorpus. NFCorpus is the development set of the project. SciFact and FiQA are held-out sets for the final run. This notebook does not load them.

In [ ]:
# On a new environment, remove the "#" from the next line.
# %pip install -q datasets==5.1.0 sentence-transformers==6.1.0 torch==2.14.1 scikit-learn==1.9.1 scipy==1.18.1 pandas==3.0.6 numpy==2.5.4 matplotlib==3.11.2

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # On macOS, this prevents an OpenMP conflict. It has no effect on other systems.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10  # Number of top documents (k) for k-occurrence
SPLITS = ["train", "validation", "test"]

# NFCorpus at fixed Hub revisions. Fixed revisions make the numbers reproducible.
NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODEL = dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0")
MAX_SEQ = 512

N_PROJ_DOCS = 3000    # Number of sampled documents for the PCA and t-SNE picture
N_RANDOM_DOCS = 1000  # Number of random documents for each query (cosine baseline)
N_TRUST = 2000        # Number of queries for the trustworthiness test

## Helpers and data

This section defines the functions that load the data, encode the text, and find the top documents for k-occurrence. The notebook uses all labelled queries from the train, validation, and test qrels. If a query is in more than one split, the notebook keeps the first split.

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def unit(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
enc = SentenceTransformer(MODEL["repo"], revision=MODEL["revision"], device="cpu")
enc.max_seq_length = MAX_SEQ

# Keep only the queries that have one or more relevant documents. The benchmark uses the same rule.
qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]
D = encode(enc, docs)
Q = encode(enc, [queries[q] for q in qids])
sp = np.array([split_of[q] for q in qids])
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
print(f"{len(docs)} docs, {len(qids)} queries", {s: int((sp == s).sum()) for s in SPLITS}, "embedding dim", D.shape[1])

# Two numbers that the text cites: relevant documents per query, and the fraction of documents relevant to a train query.
n_rel = np.array([len(r) for r in rel])
train_rel = set().union(*(r for r, s in zip(rel, sp) if s == "train"))
print(f"judged-relevant documents per query: mean {n_rel.mean():.1f}")
print(f"fraction of documents relevant to one or more train queries: {np.mean([d in train_rel for d in doc_ids]):.3f}")

## Query-document separation

For each query, this section compares two values: the mean cosine to the relevant documents of the query, and the mean cosine to 1,000 random documents. The difference (gap) shows how much nearer the relevant documents are to the query than random documents are.

This section also calculates two nearest-neighbour cosines: from each query to the other queries, and from each document to the other documents. These values show how tightly each set clusters. The document calculation uses all documents. A subsample gives each document fewer candidates, and this decreases the value.

**Why this test**
- **Estimand:** the mean over queries of (mean cosine to judged-relevant documents minus mean cosine to random documents). This value is descriptive. This section does not do a statistical test. The gap is large relative to its spread, and no decision uses a rejection of zero.
- **Unit of independence:** the query. Each query has approximately 41 judged-relevant documents, and this number includes level-1 indirect links. Because of this, the mean is lower than the cosine of the best match.
- **Assumptions:** none. This value is a summary of the geometry.
- **Why not the usual alternative:** at n = 3,237, a paired test of relevant against random cosine will always reject. It gives no new information.
- **Resolution / Monte-Carlo error:** 1,000 random documents for each query. The random-document mean for each query has an error of approximately ±0.003.

In [ ]:
def nn_sim(A, B=None, chunk=1000):
    """Return the maximum cosine from each row of A to B. If B is None, use the other rows of A. All vectors must have unit length."""
    self_ref = B is None
    B = A if self_ref else B
    best = np.empty(len(A), dtype=np.float32)
    for i in range(0, len(A), chunk):
        S = A[i : i + chunk] @ B.T
        if self_ref:
            S[np.arange(len(S)), np.arange(i, i + len(S))] = -np.inf
        best[i : i + chunk] = S.max(axis=1)
    return best


rng = np.random.default_rng(SEED)
pos = {d: i for i, d in enumerate(doc_ids)}
rel_cos, rand_cos = [], []
for qi, r in enumerate(rel):
    idx = [pos[d] for d in r if d in pos]
    rel_cos.append(float((D[idx] @ Q[qi]).mean()))
    rand_cos.append(float((D[rng.choice(len(D), N_RANDOM_DOCS, replace=False)] @ Q[qi]).mean()))
rel_cos, rand_cos = np.array(rel_cos), np.array(rand_cos)

qq = nn_sim(Q)
dd = nn_sim(D)   # Use all documents. A subsample decreases the nearest-neighbour similarity.
display(pd.DataFrame([{"queries_with_relevant": len(rel_cos), "cos_query_relevant": rel_cos.mean(), "cos_query_random": rand_cos.mean(),
                       "gap": (rel_cos - rand_cos).mean(), "nn_query_query_median": np.median(qq), "nn_doc_doc_median": np.median(dd)}]).round(3))

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(qq, bins=40, alpha=0.6, density=True, label="query to nearest query")
ax.hist(dd, bins=40, alpha=0.6, density=True, label="doc to nearest doc")
ax.set_xlabel("cosine")
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## PCA variance

This section shows the cumulative variance that the first 100 principal components of the document embeddings explain. Read the value at 2 components to find how much a 2-D picture can show. Read the values at 10 to 100 components to find how concentrated the variance is. A steep curve shows that few dimensions hold most of the variance.

In [ ]:
pca_full = PCA(n_components=100, random_state=SEED).fit(D)
cum = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.arange(1, len(cum) + 1), cum)
ax.set_xlabel("components")
ax.set_ylabel("cumulative explained variance")
fig.tight_layout()
plt.show()
pd.DataFrame([{f"var@{n}": float(cum[n - 1]) for n in (2, 10, 50, 100)}]).round(3)

## The t-SNE picture

This section makes one example map. It uses 3,000 documents and all queries, PCA to 50 dimensions, and then t-SNE (perplexity 30). Black points are documents that are relevant to one or more queries. Use this map only as an illustration. The next section tests how much you can trust it.

In [ ]:
relevant_any = set().union(*rel)
rng = np.random.default_rng(SEED)
sample = rng.choice(len(D), N_PROJ_DOCS, replace=False)
X50 = PCA(n_components=50, random_state=SEED).fit_transform(np.vstack([D[sample], Q]))
Z = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(X50)
Zd, Zq = Z[: len(sample)], Z[len(sample):]
is_rel = np.array([doc_ids[i] in relevant_any for i in sample])

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.scatter(*Zd[~is_rel].T, s=3, c="lightgrey", label="doc")
ax.scatter(*Zd[is_rel].T, s=5, c="black", label="doc (relevant to some query)")
for s, c in {"train": "tab:blue", "validation": "tab:green", "test": "tab:red"}.items():
    ax.scatter(*Zq[sp == s].T, s=10, c=c, alpha=0.7, label=f"query {s}")
ax.set_xticks([]); ax.set_yticks([])
ax.legend(markerscale=2, fontsize=8)
fig.tight_layout()
plt.show()

## t-SNE trustworthiness

Trustworthiness (0 to 1, `n_neighbors = 10`) is the fraction of 2-D neighbours that are also neighbours in the original space. The value is lower when the map has false neighbours. This section runs t-SNE on a subsample of 2,000 queries, after PCA to 50 dimensions, at three perplexities. A plain 2-D PCA is the baseline.

With `init="pca"`, the run is deterministic and the seed has no effect. For this reason, this section uses random initialization with three seeds to measure the **spread between seeds**. It also calculates the Jaccard overlap of the 10 nearest neighbours of each point between maps from different seeds. This overlap shows how much of a picture comes from the algorithm and not from the data.

**Why this test**
- **Estimand:** two values. Trustworthiness shows how well a 2-D map keeps the 10 nearest neighbours of each point. The neighbour overlap between seeds shows how much the map changes with the random seed.
- **Unit of independence:** the t-SNE run (seed). Three seeds give only an approximate spread.
- **Assumptions:** trustworthiness at k = 10 measures only local structure. It gives no information about the distances between clusters.
- **Why not the usual alternative:** a visual check of a map, or trustworthiness alone, does not show the seed dependence. The overlap between seeds shows how much of a picture is reproducible. 2-D PCA is a weak baseline, so a better value than PCA is a low requirement.
- **Resolution / Monte-Carlo error:** 3 seeds for each perplexity. The SD over seeds is an approximate value.

In [ ]:
def knn_sets(Z, k=10):
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]
    return [set(r) for r in idx]


def knn_overlap(Za, Zb):
    A, B_ = knn_sets(Za), knn_sets(Zb)
    return float(np.mean([len(a & b) / len(a | b) for a, b in zip(A, B_)]))


sub = np.random.default_rng(SEED).choice(len(Q), min(N_TRUST, len(Q)), replace=False)
X50q = PCA(50, random_state=SEED).fit_transform(Q[sub])
pca2 = PCA(2, random_state=SEED).fit_transform(Q[sub])

rows = [{"method": "PCA 2D", "init": "-", "perplexity": np.nan, "trust_mean": trustworthiness(Q[sub], pca2, n_neighbors=10),
         "trust_sd_over_seeds": np.nan, "knn_overlap_between_seeds": np.nan}]
for perp in (15, 30, 50):
    Zp = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(X50q)
    rows.append({"method": "t-SNE", "init": "pca", "perplexity": perp, "trust_mean": trustworthiness(Q[sub], Zp, n_neighbors=10),
                 "trust_sd_over_seeds": np.nan, "knn_overlap_between_seeds": np.nan})
    Zs = [TSNE(2, perplexity=perp, init="random", random_state=seed).fit_transform(X50q) for seed in range(3)]
    t = [trustworthiness(Q[sub], Z_, n_neighbors=10) for Z_ in Zs]
    ov = [knn_overlap(Zs[i], Zs[j]) for i in range(3) for j in range(i + 1, 3)]
    rows.append({"method": "t-SNE", "init": "random (3 seeds)", "perplexity": perp, "trust_mean": np.mean(t),
                 "trust_sd_over_seeds": np.std(t, ddof=1), "knn_overlap_between_seeds": np.mean(ov)})
pd.DataFrame(rows).round(4)

## Hubness against null models (equal sizes)

k-occurrence `N_10` is the number of queries that have a given document in their exact top 10 by inner product. This section uses two hubness statistics: the skewness of `N_10`, and the fraction of documents that no query retrieves.

**Design.** The notebook takes 20 subsamples of 3,000 documents and 1,000 queries. Thus, the ratio of queries to documents does not change. The interval for the real data is the 2.5 to 97.5 percentile across the subsamples. This interval shows subsampling variability. It is not a confidence interval, because each subsample includes approximately 83% of the corpus.

**Nulls at the same sizes (20 draws each).** The interval for each null is the 2.5 to 97.5 percentile across its 20 draws.
- *Isotropic*: unit vectors with a uniformly random direction in 384 dimensions.
- *Covariance-matched Gaussian*: documents ~ N(mean_D, cov_D) and queries ~ N(mean_Q, cov_Q), normalized after the draw. The notebook draws the queries and the documents **independently**. This null keeps the anisotropy and the query-document offset. It removes all higher-order (cluster) structure **and all query-document alignment**.

**How to read the result.** If the real value is higher than the covariance-matched value, semantic structure adds hubness beyond anisotropy. A real value equal to or lower than the covariance-matched value shows only one thing. The real data is *not more hub-prone than an anisotropic Gaussian with no query-document coupling*. This result does not show that topic structure has no effect.

**Mean-centering** is the standard hubness remedy. It subtracts the document mean from both sets and normalizes again. It shows how much of the real hubness comes from proximity to the centroid.

**Why this test**
- **Estimand:** the skewness of k-occurrence (and the fraction never retrieved) for this encoder on this corpus at fixed sizes. The section compares this value with the same statistic under two generative nulls.
- **Unit of independence:** the null draw. The real-data subsamples are not independent replicates. Thus, the section calculates no p-value. It compares the real value with the spread of each null.
- **Assumptions:** the covariance-matched null assumes that only the first and second moments are necessary to match, separately for queries and documents.
- **Why not the usual alternative:** a comparison with an absolute threshold, or with only an isotropic null, will incorrectly show usual anisotropy as "semantic hubness". A test of topic structure needs a null that also keeps the query-document alignment. This notebook does not calculate that null, so it makes no claim about topic structure.
- **Resolution / Monte-Carlo error:** 20 draws for each null. The 2.5 to 97.5 percentile range is only approximate, because the extreme percentiles of 20 draws are near the minimum and the maximum.

In [ ]:
def kocc(Q_, D_, k=K, chunk=500):
    counts = np.zeros(len(D_), dtype=np.int64)
    for i in range(0, len(Q_), chunk):
        S = Q_[i : i + chunk] @ D_.T
        top = np.argsort(-S, axis=1, kind="stable")[:, :k]   # Stable sort: if two scores are equal, the lower document index comes first.
        np.add.at(counts, top.ravel(), 1)
    return counts


def hub_stats(c):
    return stats.skew(c), float((c == 0).mean())


def gauss_like(X, n, rng):
    mu = X.mean(0)
    w, V = np.linalg.eigh(np.cov(X, rowvar=False))
    w = np.clip(w, 0, None)
    return unit(mu + (rng.standard_normal((n, X.shape[1])) * np.sqrt(w)) @ V.T)


ND, NQ = 3000, 1000
N_SUB, N_NULL = 20, 20      # Number of real-data subsamples; number of draws for each null model
rng = np.random.default_rng(SEED)
real, centered = [], []
for _ in range(N_SUB):
    Di = D[rng.choice(len(D), ND, replace=False)]
    Qi = Q[rng.choice(len(Q), NQ, replace=False)]
    real.append(hub_stats(kocc(Qi, Di)))
    mu = Di.mean(0)
    centered.append(hub_stats(kocc(unit(Qi - mu), unit(Di - mu))))
iso = [hub_stats(kocc(unit(rng.standard_normal((NQ, D.shape[1]))), unit(rng.standard_normal((ND, D.shape[1]))))) for _ in range(N_NULL)]
cov = [hub_stats(kocc(gauss_like(Q, NQ, rng), gauss_like(D, ND, rng))) for _ in range(N_NULL)]

rows = []
for label, arr in (("real", real), ("real, mean-centered", centered), ("isotropic null", iso), ("covariance-matched null", cov)):
    a = np.array(arr)
    rows.append({"version": label, "draws": len(a), "skew_mean": a[:, 0].mean(), "skew_lo": np.percentile(a[:, 0], 2.5),
                 "skew_hi": np.percentile(a[:, 0], 97.5), "never_retrieved_mean": a[:, 1].mean(),
                 "never_lo": np.percentile(a[:, 1], 2.5), "never_hi": np.percentile(a[:, 1], 97.5)})
pd.DataFrame(rows).round(3)

## Correlates of k-occurrence

This section calculates the Spearman correlation between the k-occurrence of a document and two values: its **cosine to the document centroid** (the usual hubness driver) and its **length in words**. k-occurrence uses all labelled queries, the exact top 10, and the full corpus. The 95% bootstrap CIs resample documents. This bootstrap treats documents as independent. This is optimistic, because the corpus has near-duplicate documents.

**Why this test**
- **Estimand:** the rank correlation between the k-occurrence of a document and each candidate driver.
- **Unit of independence:** the document. This is optimistic, because near-duplicates make the CI too narrow.
- **Assumptions:** a monotone relation. k-occurrence is a skewed count with many zeros. Thus, Spearman is better than Pearson here.
- **Why not the usual alternative:** a regression on both drivers needs a count model. It also mixes in a second effect. Of the documents, 98.8% are relevant to one or more train queries, so a frequently retrieved document can be a correct hub. The next step is to split k-occurrence into relevant and non-relevant hits. This notebook does not do this step.
- **Resolution / Monte-Carlo error:** 300 bootstrap draws. The CI limits have an error of approximately ±0.01.

In [ ]:
c = kocc(Q, D)
centroid = D.mean(0)
cent_cos = D @ (centroid / np.linalg.norm(centroid))
words = np.array([len(t.split()) for t in docs])

rng = np.random.default_rng(SEED)
rows = []
for label, x in (("cosine to centroid", cent_cos), ("words", words)):
    rho = stats.spearmanr(x, c)[0]
    b = [stats.spearmanr(x[i], c[i])[0] for i in (rng.integers(0, len(x), len(x)) for _ in range(300))]
    rows.append({"driver": label, "spearman": rho, "ci_lo": np.percentile(b, 2.5), "ci_hi": np.percentile(b, 97.5)})
pd.DataFrame(rows).round(3)

In [ ]:
import datasets, matplotlib, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "max_seq_length": MAX_SEQ})

## Findings
All numbers come from one full run (seed 0, NFCorpus, multi-qa-MiniLM, 3,633 documents, 3,237 queries, 384 dimensions).

- **No 2-D picture is faithful, but the variance is concentrated.** The first 2 principal components explain 8.8% of the document variance. Thus, a 2-D map cannot show most of the structure. But the variance is not spread evenly: 10 of 384 components explain 27.9%, 50 explain 64.4%, and 100 explain 83.5%.
- **Relevant documents are only a small distance nearer than random documents.** The mean cosine from a query to its judged-relevant documents is 0.20. The mean cosine to random documents is 0.09 (gap 0.11). Each query has approximately 41 judged-relevant documents, and the mean includes all of them. The median nearest-neighbour cosine is 0.702 for queries and 0.712 for documents.
- **t-SNE keeps local neighbourhoods.** Trustworthiness is 0.91 to 0.92, against 0.71 for 2-D PCA. The score is stable between seeds (SD 0.001 to 0.002). But only 44% to 51% of the 10 nearest neighbours of each point are the same between random-init seeds.
- **Hubness is higher than the isotropic null but lower than the covariance-matched null.** The skewness of k-occurrence is 1.70 (subsample range 1.49 to 1.90). The isotropic null gives 0.54 (range 0.47 to 0.61). The covariance-matched Gaussian gives 2.62 (range 2.21 to 3.16). Each null has 20 draws. No query retrieves 11.6% of the documents (17.4% under the covariance-matched null). Mean-centering *increased* the skewness to 3.70. This is the opposite of the usual effect.
- **Hub drivers.** k-occurrence has a negative correlation with the cosine to the document centroid (Spearman -0.26, CI -0.29 to -0.23). Thus, queries retrieve documents that are farther from the centroid more frequently. k-occurrence also has a small negative correlation with length (-0.16, CI -0.19 to -0.13).

### What this supports
- For this model on this corpus, the embedding space is anisotropic. Its variance is concentrated in the first 10 to 50 of 384 components.
- No 2-D map shows this space faithfully. The 2-D maps keep local neighbourhoods, but they do not keep the cluster layout.
- Hubness is present. The real data is not more hub-prone than an anisotropic Gaussian with no query-document coupling.

### What this does not support
- That geometry alone *causes* hubness, or that topic structure has no effect. The covariance-matched null draws queries and documents independently, so it has no query-document alignment. This notebook does not calculate a null that keeps the alignment.
- That hubs are harmful for retrieval. This notebook does not test this. In NFCorpus, 98.8% of documents are relevant to one or more train queries, so a frequently retrieved document can be a correct hub. This notebook does not split k-occurrence into relevant and non-relevant hits. Thus, the project must not use hubs as a failure signal yet.
- That 2-D pictures show real cluster structure or real distances.
- That the mean-centering result is general. Here, the remedy *increased* the skewness. This notebook does not test the cause.
- A result for other encoders, other corpora, or the held-out datasets.
- That the subsample range for skewness is a confidence interval. Each subsample includes approximately 83% of the corpus. The 2.5 to 97.5 percentile of 20 null draws is also only an approximate range.